# Computer Vision Deep Learning Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/njpinton/CMSC178IP/blob/main/09-ComputerVisionDeepLearningI/notebooks/cv_deep_learning_workshop.ipynb)

---

## Workshop Objectives

By the end of this 60-minute workshop, you will:

1. Understand the fundamentals of neural networks for computer vision
2. Implement basic building blocks: perceptrons and activation functions
3. Build and train a Convolutional Neural Network (CNN) from scratch
4. Visualize and diagnose model performance
5. Apply best practices to improve model accuracy
6. Gain hands-on experience with real image datasets

**Duration:** 45-60 minutes

**Prerequisites:** Basic Python, NumPy knowledge

---

## Setup & Imports

Let's prepare our environment with all necessary libraries.

In [ ]:
# Core libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Deep learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Utilities
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.style.use('default')
%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print("Setup complete!")

---

## Part 1: Problem Understanding

### Why Deep Learning for Computer Vision?

Traditional computer vision techniques (like edge detection, SIFT, HOG) require manual feature engineering. Deep learning, particularly CNNs, can **automatically learn** hierarchical features from raw pixel data:

- **Layer 1:** Edges and simple patterns
- **Layer 2:** Textures and shapes
- **Layer 3:** Object parts
- **Final layers:** Complete objects

### Our Task Today

We'll work with the **CIFAR-10 dataset**: 60,000 color images (32x32 pixels) across 10 classes:
- airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

Let's load and explore the data!

In [ ]:
# Load CIFAR-10 dataset
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Use a subset for faster training (5000 training, 1000 test)
X_train = X_train_full[:5000]
y_train = y_train_full[:5000]
X_test = X_test[:1000]
y_test = y_test[:1000]

print(f"Training set: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test set: {X_test.shape}, Labels: {y_test.shape}")
print(f"Image shape: {X_train[0].shape}")
print(f"Pixel value range: [{X_train.min()}, {X_train.max()}]")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
fig.suptitle('CIFAR-10 Sample Images', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(f"{class_names[y_train[i][0]]}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Data Preprocessing

Neural networks work best with normalized inputs. We'll:
1. Scale pixel values from [0, 255] to [0, 1]
2. Convert labels to one-hot encoding

In [ ]:
# Normalize pixel values to [0, 1]
X_train_normalized = X_train.astype('float32') / 255.0
X_test_normalized = X_test.astype('float32') / 255.0

# One-hot encode labels
y_train_categorical = to_categorical(y_train, 10)
y_test_categorical = to_categorical(y_test, 10)

print(f"Normalized pixel range: [{X_train_normalized.min():.2f}, {X_train_normalized.max():.2f}]")
print(f"Original label: {y_train[0]} -> One-hot: {y_train_categorical[0]}")

---

## Part 2: Core Methods - Building Blocks

Before diving into CNNs, let's understand the fundamental building blocks.

### 2.1 The Perceptron

A perceptron is the simplest neural network unit:

$$y = f(\sum_{i=1}^{n} w_i x_i + b)$$

where:
- $x_i$ are inputs
- $w_i$ are weights
- $b$ is bias
- $f$ is an activation function

In [ ]:
class SimplePerceptron:
    """A basic perceptron implementation."""
    
    def __init__(self, n_inputs):
        # Initialize weights randomly and bias to zero
        self.weights = np.random.randn(n_inputs) * 0.01
        self.bias = 0.0
    
    def forward(self, x):
        """Compute weighted sum + bias."""
        return np.dot(x, self.weights) + self.bias

# Test the perceptron
perceptron = SimplePerceptron(n_inputs=3)
test_input = np.array([1.0, 2.0, 3.0])
output = perceptron.forward(test_input)

print(f"Input: {test_input}")
print(f"Weights: {perceptron.weights}")
print(f"Bias: {perceptron.bias}")
print(f"Output: {output:.4f}")

### 2.2 Activation Functions

Activation functions introduce non-linearity, allowing networks to learn complex patterns.

**Common activation functions:**
- **ReLU:** $f(x) = \max(0, x)$ - Most popular for hidden layers
- **Sigmoid:** $f(x) = \frac{1}{1 + e^{-x}}$ - Outputs between 0 and 1
- **Softmax:** $f(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$ - For multi-class classification

In [ ]:
# Implement activation functions
def relu(x):
    """Rectified Linear Unit."""
    return np.maximum(0, x)

def sigmoid(x):
    """Sigmoid activation."""
    return 1 / (1 + np.exp(-x))

def softmax(x):
    """Softmax activation for multi-class."""
    exp_x = np.exp(x - np.max(x))  # Numerical stability
    return exp_x / exp_x.sum()

# Visualize activation functions
x = np.linspace(-5, 5, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, relu(x), 'b-', linewidth=2)
axes[0].set_title('ReLU', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

axes[1].plot(x, sigmoid(x), 'r-', linewidth=2)
axes[1].set_title('Sigmoid', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

# Softmax example with 3 values
test_logits = np.array([-2, 0, 2])
test_probs = softmax(test_logits)
axes[2].bar(['Class 1', 'Class 2', 'Class 3'], test_probs, color=['#ff9999', '#66b3ff', '#99ff99'])
axes[2].set_title('Softmax Example', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Probability')
axes[2].set_ylim([0, 1])
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Softmax input: {test_logits}")
print(f"Softmax output: {test_probs}")
print(f"Sum of probabilities: {test_probs.sum():.4f}")

### 2.3 Forward Propagation

Forward propagation is how data flows through the network:

1. Input layer receives data
2. Each layer computes: activation(weights × inputs + bias)
3. Output layer produces predictions

In [ ]:
# Simple 2-layer network demonstration
class SimpleTwoLayerNet:
    """A basic two-layer neural network."""
    
    def __init__(self, input_size, hidden_size, output_size):
        # Initialize weights
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros(output_size)
    
    def forward(self, X):
        """Forward pass through the network."""
        # Layer 1
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
        
        # Layer 2 (output)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        
        return self.a2

# Test the network
net = SimpleTwoLayerNet(input_size=10, hidden_size=5, output_size=3)
test_input = np.random.randn(10)
output = net.forward(test_input)

print(f"Input size: {test_input.shape}")
print(f"Hidden layer size: {net.a1.shape}")
print(f"Output probabilities: {output}")
print(f"Predicted class: {np.argmax(output)}")

---

## Part 3: Advanced Techniques - Convolutional Neural Networks

### Why CNNs for Images?

Fully connected networks don't work well for images because:
- Too many parameters (32×32×3 = 3,072 inputs per image!)
- Ignore spatial structure
- Don't scale to larger images

**CNNs solve this with:**
1. **Convolution:** Local feature detection
2. **Pooling:** Dimensionality reduction
3. **Parameter sharing:** Fewer weights to learn

### 3.1 Understanding Convolution

In [ ]:
# Visualize how convolution works
def visualize_convolution():
    """Demonstrate convolution operation on a simple image."""
    
    # Create a simple 7x7 image with a vertical edge
    image = np.zeros((7, 7))
    image[:, 3:] = 1.0
    
    # Vertical edge detector kernel
    kernel_vertical = np.array([[-1, 0, 1],
                                [-1, 0, 1],
                                [-1, 0, 1]])
    
    # Horizontal edge detector kernel
    kernel_horizontal = np.array([[-1, -1, -1],
                                  [ 0,  0,  0],
                                  [ 1,  1,  1]])
    
    # Apply convolution manually
    def convolve2d(image, kernel):
        k_height, k_width = kernel.shape
        i_height, i_width = image.shape
        output_height = i_height - k_height + 1
        output_width = i_width - k_width + 1
        output = np.zeros((output_height, output_width))
        
        for i in range(output_height):
            for j in range(output_width):
                output[i, j] = np.sum(image[i:i+k_height, j:j+k_width] * kernel)
        
        return output
    
    result_vertical = convolve2d(image, kernel_vertical)
    result_horizontal = convolve2d(image, kernel_horizontal)
    
    # Visualize
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    # Row 1: Vertical edge detection
    axes[0, 0].imshow(image, cmap='gray')
    axes[0, 0].set_title('Input Image', fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(kernel_vertical, cmap='RdBu', vmin=-1, vmax=1)
    axes[0, 1].set_title('Vertical Edge Kernel', fontweight='bold')
    for i in range(3):
        for j in range(3):
            axes[0, 1].text(j, i, f'{kernel_vertical[i,j]:.0f}', 
                          ha='center', va='center', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(result_vertical, cmap='gray')
    axes[0, 2].set_title('Vertical Edge Detected', fontweight='bold')
    axes[0, 2].axis('off')
    
    # Row 2: Horizontal edge detection
    axes[1, 0].imshow(image, cmap='gray')
    axes[1, 0].set_title('Input Image', fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(kernel_horizontal, cmap='RdBu', vmin=-1, vmax=1)
    axes[1, 1].set_title('Horizontal Edge Kernel', fontweight='bold')
    for i in range(3):
        for j in range(3):
            axes[1, 1].text(j, i, f'{kernel_horizontal[i,j]:.0f}', 
                          ha='center', va='center', fontsize=12)
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(result_horizontal, cmap='gray')
    axes[1, 2].set_title('Horizontal Edge Detected', fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_convolution()

### 3.2 Building a CNN with Keras

Now let's build a complete CNN architecture:

```
Input (32×32×3)
    ↓
Conv2D (32 filters, 3×3) + ReLU
    ↓
MaxPooling (2×2)
    ↓
Conv2D (64 filters, 3×3) + ReLU
    ↓
MaxPooling (2×2)
    ↓
Flatten
    ↓
Dense (64) + ReLU
    ↓
Dense (10) + Softmax
```

In [ ]:
def create_cnn_model():
    """Build a CNN for CIFAR-10 classification."""
    
    model = models.Sequential([
        # First convolutional block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3), 
                     padding='same', name='conv1'),
        layers.MaxPooling2D((2, 2), name='pool1'),
        
        # Second convolutional block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'),
        layers.MaxPooling2D((2, 2), name='pool2'),
        
        # Third convolutional block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv3'),
        
        # Fully connected layers
        layers.Flatten(name='flatten'),
        layers.Dense(64, activation='relu', name='fc1'),
        layers.Dense(10, activation='softmax', name='output')
    ])
    
    return model

# Create and compile the model
model = create_cnn_model()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

### 3.3 Training the CNN

Time to train our model! We'll:
- Use 10% of training data for validation
- Train for 10 epochs
- Use batch size of 64

In [ ]:
# Train the model
history = model.fit(
    X_train_normalized, y_train_categorical,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

---

## Part 4: Diagnostic Tools - Visualization & Interpretation

### 4.1 Training History Visualization

Plotting loss and accuracy curves helps us understand model learning.

In [ ]:
def plot_training_history(history):
    """Visualize training and validation metrics."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot loss
    axes[0].plot(history.history['loss'], 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].legend(loc='upper right', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Plot accuracy
    axes[1].plot(history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

### 4.2 Model Evaluation

Let's evaluate our model on the test set.

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_normalized, y_test_categorical, verbose=0)

print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

### 4.3 Confusion Matrix

A confusion matrix shows which classes the model confuses with each other.

In [ ]:
# Generate predictions
y_pred_probs = model.predict(X_test_normalized, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

### 4.4 Visualizing Feature Maps

Let's see what the CNN "sees" at different layers.

In [ ]:
def visualize_feature_maps(model, image, layer_names):
    """Visualize feature maps from intermediate layers."""
    
    # Create a model that outputs feature maps
    layer_outputs = [model.get_layer(name).output for name in layer_names]
    activation_model = models.Model(inputs=model.input, outputs=layer_outputs)
    
    # Get activations
    activations = activation_model.predict(image[np.newaxis, ...], verbose=0)
    
    # Visualize
    fig, axes = plt.subplots(len(layer_names) + 1, 8, figsize=(16, 2*(len(layer_names)+1)))
    
    # Show original image
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Input Image', fontweight='bold')
    axes[0, 0].axis('off')
    for i in range(1, 8):
        axes[0, i].axis('off')
    
    # Show feature maps
    for layer_idx, (layer_name, activation) in enumerate(zip(layer_names, activations), 1):
        n_features = min(8, activation.shape[-1])
        for i in range(n_features):
            axes[layer_idx, i].imshow(activation[0, :, :, i], cmap='viridis')
            if i == 0:
                axes[layer_idx, i].set_ylabel(layer_name, fontweight='bold', rotation=0, 
                                             labelpad=40, ha='right')
            axes[layer_idx, i].axis('off')
        for i in range(n_features, 8):
            axes[layer_idx, i].axis('off')
    
    plt.suptitle('CNN Feature Maps Visualization', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

# Visualize feature maps for a test image
sample_image = X_test_normalized[0]
visualize_feature_maps(model, sample_image, ['conv1', 'conv2', 'conv3'])

### 4.5 Prediction Visualization

Let's see how the model performs on individual images.

In [ ]:
def visualize_predictions(model, images, true_labels, n_samples=10):
    """Visualize model predictions with confidence scores."""
    
    predictions = model.predict(images[:n_samples], verbose=0)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for i in range(n_samples):
        ax = axes[i]
        
        # Display image
        ax.imshow(images[i])
        
        # Get prediction
        pred_class = np.argmax(predictions[i])
        pred_prob = predictions[i][pred_class]
        true_class = true_labels[i][0]
        
        # Set title with color coding
        is_correct = pred_class == true_class
        color = 'green' if is_correct else 'red'
        
        title = f"True: {class_names[true_class]}\n"
        title += f"Pred: {class_names[pred_class]} ({pred_prob*100:.1f}%)"
        ax.set_title(title, color=color, fontsize=9, fontweight='bold')
        ax.axis('off')
    
    plt.suptitle('Model Predictions (Green=Correct, Red=Incorrect)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_predictions(model, X_test_normalized, y_test, n_samples=10)

---

## Part 5: Best Practices

### 5.1 Detecting Overfitting

**Signs of overfitting:**
- Training accuracy much higher than validation accuracy
- Validation loss increases while training loss decreases

**Solutions:**
1. Add dropout layers
2. Use data augmentation
3. Reduce model complexity
4. Add L2 regularization

### 5.2 Data Augmentation

Artificially expand your dataset by applying random transformations.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create data augmentation generator
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Visualize augmented images
sample_image = X_train_normalized[0]
sample_image_batch = sample_image[np.newaxis, ...]  # Add batch dimension

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

# Original image
axes[0].imshow(sample_image)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

# Augmented versions
aug_iter = datagen.flow(sample_image_batch, batch_size=1)
for i in range(1, 10):
    aug_image = next(aug_iter)[0]
    axes[i].imshow(aug_image)
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Improved Model with Regularization

Let's create an improved model with dropout and batch normalization.

In [ ]:
def create_improved_cnn():
    """Build an improved CNN with regularization."""
    
    model = models.Sequential([
        # First block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Fully connected
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    return model

improved_model = create_improved_cnn()
improved_model.compile(optimizer='adam', 
                      loss='categorical_crossentropy', 
                      metrics=['accuracy'])

print("Improved model created with:")
print("- Batch Normalization for stable training")
print("- Dropout for regularization")
print("- Deeper architecture for better feature learning")

### 5.4 Hyperparameter Tuning Tips

**Key hyperparameters to tune:**

1. **Learning rate:** Start with 0.001, adjust if loss plateaus
2. **Batch size:** 32-128, larger = faster but less stable
3. **Number of filters:** 32, 64, 128 in progressive layers
4. **Dropout rate:** 0.25-0.5 for different layers
5. **Optimizer:** Adam (good default), SGD+momentum, RMSprop

**General tips:**
- Start simple, add complexity gradually
- Monitor validation metrics closely
- Use early stopping to prevent overfitting
- Save best model checkpoints

---

## Part 6: Student Activity (15 minutes)

### Challenge: Build and Optimize Your Own CNN

**Objective:** Train a CNN that achieves **>60% test accuracy** on CIFAR-10

**Your Tasks:**

1. **Build a CNN** with at least:
   - 2 convolutional blocks
   - Batch normalization
   - Dropout layers
   - At least 1 fully connected layer

2. **Train the model** using:
   - Data augmentation (provided below)
   - Early stopping
   - 15-20 epochs

3. **Evaluate and visualize:**
   - Plot training history
   - Generate confusion matrix
   - Report final test accuracy

**Hints:**
- Use filter sizes: 32 → 64 → 128
- Add dropout after pooling layers (0.25) and dense layers (0.5)
- Use 'adam' optimizer with default learning rate
- Batch size of 64 works well

### Starter Code

In [ ]:
# Data augmentation setup (use this!)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Callbacks for better training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

In [ ]:
# TODO: Build your model here
# Hint: Start with create_improved_cnn() and modify it

student_model = models.Sequential([
    # YOUR CODE HERE
    # Add convolutional layers, batch norm, pooling, dropout
    # ...
])

# TODO: Compile your model
# student_model.compile(...)

# TODO: Display model summary
# student_model.summary()

In [ ]:
# TODO: Train your model with data augmentation
# Hint: Use train_datagen.flow() and model.fit()

# student_history = student_model.fit(
#     train_datagen.flow(...),
#     epochs=...,
#     validation_data=...,
#     callbacks=callbacks
# )

In [ ]:
# TODO: Evaluate your model
# Use the visualization functions from Part 4

# 1. Plot training history
# plot_training_history(student_history)

# 2. Evaluate on test set
# test_loss, test_accuracy = student_model.evaluate(...)

# 3. Generate confusion matrix
# ...

### Reflection Questions

Answer these after completing your model:

1. What was your final test accuracy?
2. Did you observe overfitting? How did you address it?
3. Which classes did your model confuse most often?
4. What architectural choices improved performance most?
5. If you had more time, what would you try next?

---

## Part 7: Solutions

### Complete Solution to Student Activity

In [ ]:
# Solution: Build an optimized CNN
def create_solution_cnn():
    """Solution model with best practices."""
    
    model = models.Sequential([
        # Block 1: 32 filters
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2: 64 filters
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3: 128 filters
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Fully connected
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    return model

solution_model = create_solution_cnn()
solution_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

solution_model.summary()

In [ ]:
# Solution: Train with data augmentation
solution_history = solution_model.fit(
    train_datagen.flow(X_train_normalized, y_train_categorical, batch_size=64),
    steps_per_epoch=len(X_train_normalized) // 64,
    epochs=20,
    validation_data=(X_test_normalized, y_test_categorical),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Solution: Evaluate the model
plot_training_history(solution_history)

# Test set evaluation
test_loss, test_accuracy = solution_model.evaluate(X_test_normalized, y_test_categorical, verbose=0)
print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Final Test Loss: {test_loss:.4f}")

In [ ]:
# Solution: Confusion matrix
y_pred_probs_solution = solution_model.predict(X_test_normalized, verbose=0)
y_pred_solution = np.argmax(y_pred_probs_solution, axis=1)
y_true_solution = y_test.flatten()

cm_solution = confusion_matrix(y_true_solution, y_pred_solution)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_solution, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Solution Model - Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nPer-Class Performance:")
print(classification_report(y_true_solution, y_pred_solution, target_names=class_names))

In [ ]:
# Solution: Visualize predictions
visualize_predictions(solution_model, X_test_normalized, y_test, n_samples=10)

### Key Insights from Solution

**Architecture choices:**
1. **Progressive filter growth** (32→64→128): Captures increasingly complex features
2. **Batch normalization**: Stabilizes training and speeds convergence
3. **Strategic dropout**: 0.25 after conv blocks, 0.5 before output prevents overfitting
4. **Multiple conv layers per block**: Deeper feature learning

**Training strategies:**
1. **Data augmentation**: Effectively increases dataset size
2. **Early stopping**: Prevents overfitting by monitoring validation loss
3. **Learning rate reduction**: Adapts when plateauing

**Expected performance:**
- Test accuracy: 60-70%
- Common confusions: cat↔dog, automobile↔truck, bird↔airplane
- Training time: ~5-10 minutes on CPU, <2 minutes on GPU

---

## Part 8: Summary & Key Takeaways

### What We Learned

1. **Neural Network Fundamentals**
   - Perceptrons compute weighted sums with activation functions
   - Forward propagation flows data through layers
   - Activation functions (ReLU, softmax) introduce non-linearity

2. **Convolutional Neural Networks**
   - Convolution layers detect local features
   - Pooling reduces dimensionality
   - Parameter sharing makes CNNs efficient for images

3. **Training & Optimization**
   - Data normalization improves training
   - Data augmentation prevents overfitting
   - Regularization (dropout, batch norm) improves generalization

4. **Diagnostic Tools**
   - Loss/accuracy curves show learning progress
   - Confusion matrices reveal class-specific performance
   - Feature maps visualize learned representations

### Best Practices Checklist

- [ ] Normalize input data
- [ ] Use appropriate activation functions (ReLU for hidden, softmax for output)
- [ ] Add batch normalization for stable training
- [ ] Include dropout to prevent overfitting
- [ ] Apply data augmentation for small datasets
- [ ] Monitor both training and validation metrics
- [ ] Use callbacks (early stopping, learning rate scheduling)
- [ ] Visualize results (confusion matrix, predictions)

### Next Steps

**To improve your models further:**

1. **Transfer Learning**: Use pre-trained models (ResNet, VGG, EfficientNet)
2. **Advanced Architectures**: Explore ResNets, DenseNets, Vision Transformers
3. **Hyperparameter Tuning**: Systematic search with tools like Keras Tuner
4. **Ensemble Methods**: Combine multiple models for better accuracy
5. **More Data**: Use full CIFAR-10 or try ImageNet

**Resources:**
- [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)
- [CS231n: CNNs for Visual Recognition](http://cs231n.stanford.edu/)
- [Deep Learning Book](https://www.deeplearningbook.org/)
- [Papers with Code](https://paperswithcode.com/)

### Final Challenge

Try applying what you learned to a new dataset:
- Fashion-MNIST (clothing classification)
- CIFAR-100 (100 fine-grained classes)
- Your own image dataset!

---

**Congratulations on completing the Computer Vision Deep Learning Workshop!**

## Bonus: Quick Reference

### Common CNN Layer Patterns

```python
# Basic conv block
layers.Conv2D(filters, (3, 3), activation='relu', padding='same')
layers.MaxPooling2D((2, 2))

# Conv block with batch norm and dropout
layers.Conv2D(filters, (3, 3), activation='relu', padding='same')
layers.BatchNormalization()
layers.MaxPooling2D((2, 2))
layers.Dropout(0.25)

# Dense block with regularization
layers.Dense(units, activation='relu')
layers.BatchNormalization()
layers.Dropout(0.5)
```

### Typical Hyperparameter Ranges

| Hyperparameter | Typical Range | Notes |
|---------------|---------------|-------|
| Learning rate | 0.0001 - 0.01 | Start with 0.001 |
| Batch size | 32 - 128 | Powers of 2 |
| Dropout rate | 0.2 - 0.5 | Higher for larger layers |
| Conv filters | 32, 64, 128, 256 | Double each block |
| Dense units | 64 - 512 | Decrease toward output |

### Common Issues & Fixes

| Problem | Likely Cause | Solution |
|---------|--------------|----------|
| Loss not decreasing | Learning rate too high/low | Adjust learning rate |
| Training acc >> Val acc | Overfitting | Add dropout, regularization |
| Training very slow | Batch size too small | Increase batch size |
| Poor accuracy | Model too simple | Add layers/filters |
| NaN loss | Exploding gradients | Reduce learning rate |
